# 05 · The dashboard: how `app/app.py` works

**Transcript Intelligence: notebook 5 of 5**

**What this is.** A guided tour of `app/app.py`, the Streamlit dashboard. The notebook imports
the dashboard's own functions and runs them on the real data, so every chart and number below is
exactly what the dashboard shows, not a re-implementation.

**The idea in one line.** *Precompute, then present.* The pipeline (`src/pipeline.py`) does all
the heavy and paid work: loading, enrichment, Claude classification and sentiment scoring. The
dashboard only reads those files. What depends on the sidebar filters (account risk, the incident
timeline, SLA, competitors, talk time) is cheap, so it's recomputed from the filtered calls on
every interaction, and every number follows the selected time period.

**How to run the dashboard.** `streamlit run app/app.py`, or `python run_app.py`, or
`TranscriptIntel.exe`. This notebook doesn't start it and makes no LLM calls.

**Contents**

1. Setup: importing the dashboard without starting it
2. Code map
3. Loading data: modes and caching
4. Filters and the time period
5. The context: one dict that feeds every tab
6. Overview tab
7. Topics tab
8. Confidence tab
9. Accounts tab
10. Incidents tab
11. Insights tab
12. Tool Telemetry tab
13. Every number follows the period
14. New transcripts, refresh and logs
15. Extending the dashboard

## 1. Setup: importing the dashboard without starting it

`streamlit run` executes `app.py` as `__main__`, and only then does the last line call `main()`.
Imported under another name, the module just defines its functions. Streamlit calls made outside a
running app (such as `st.cache_data`) fall back to a simple in-memory mode and log warnings, which
are silenced here.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, json, logging, importlib.util, inspect
from datetime import timedelta
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd
import plotly.io as pio
from IPython.display import HTML, Markdown, display

import streamlit  # noqa: F401  (create Streamlit's loggers first so they can be silenced)
for name in list(logging.root.manager.loggerDict):
    if name.startswith("streamlit"):
        logging.getLogger(name).setLevel(logging.ERROR)

spec = importlib.util.spec_from_file_location("dashboard", ROOT / "app" / "app.py")
app = importlib.util.module_from_spec(spec)
spec.loader.exec_module(app)

pio.renderers.default = "plotly_mimetype+notebook_connected"
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
print("Imported", spec.origin)

## 2. Code map

`app.py` is one file, organised top to bottom in the order the page is built:

| Section | Functions | Job |
|---|---|---|
| Constants | `CALL_TYPE_COLORS`, `TIER_COLORS`, `SCAN_INTERVALS`, `DATE_FMT`, … | Fixed colours per call type, status colours reserved for state, mm/dd/yyyy formats |
| Loading | `available_modes`, `load_base`, `load_mode`, `merged_calls` | Read the pipeline's files, cached |
| Header and data | `config_popover`, `run_refresh`, `transcript_watch`, `data_section` | ⚙ Config, the new-transcript check, the Refresh button and the pipeline log |
| Filters | `quick_ranges`, `sidebar_filters`, `filter_calls` | Time period, call type, product and account |
| Helpers | `kpi`, `show_table`, `bar_list`, `badge`, `money`, `fmt_ts`, … | Small HTML pieces and table formatting |
| Narrative | `headline_insight`, `accomplishments`, `improvements` | The Overview's sentences, generated from the data |
| Charts | `*_chart`, `account_heatmap`, `theme_period_heatmap` | One Plotly figure each, no Streamlit calls inside |
| Tabs | `render_overview` … `render_telemetry`, `account_brief`, the two dialogs | Lay out each tab from the context |
| Entry point | `build_context`, `main` | Load, filter, compute the context, draw the tabs |

The chart and narrative functions take plain DataFrames and return a figure or a string. That
separation is what lets this notebook reuse them.

In [ ]:
own = {n: f for n, f in inspect.getmembers(app, inspect.isfunction) if f.__module__ == "dashboard"}
def kind(n):
    if n.startswith("render_") or n.endswith("_dialog"): return "tab"
    if n.endswith("_chart") or n.endswith("_heatmap"): return "chart"
    if n in {"headline_insight", "accomplishments", "improvements", "account_brief"}: return "narrative"
    return "other"
pd.DataFrame([{"function": n, "kind": kind(n), "line": inspect.getsourcelines(f)[1],
               "does": (inspect.getdoc(f) or "").split("\n")[0]} for n, f in own.items()]) \
  .sort_values("line").reset_index(drop=True)

## 3. Loading data: modes and caching

Two groups of files are read:

- **`load_base()`**: tables that don't depend on the analysis mode (calls, sentences, key
  moments, action items, staff roster, sentiment scores), plus the time of the last pipeline run.
  That time doubles as a **data version**: if it changes on disk, the sidebar offers *Load latest
  data*.
- **`load_mode(mode)`**: the outputs that differ between **Hybrid (LLM + rules)** and **Rules
  only**: themes, topic clusters, the incident register and the insight tables, from
  `data/processed/<mode>/`.

Both use `@st.cache_data`, so files are read once per pipeline run, not on every click. A refresh
clears the cache. These files are written after the pipeline's screening stage, so non-business
calls (birthdays, farewells, socials) are already gone: the dashboard has no filter for them
because it never sees them (`01_explore`, section 4).

In [ ]:
modes = app.available_modes()
print("Modes built:", {m: app.MODES[m] for m in modes})

tables, call_sent, sent_metrics, version = app.load_base()
MODE = "hybrid" if "hybrid" in modes else modes[0]
call_topics, clusters, topic_metrics, register, ins = app.load_mode(MODE)
all_calls = app.merged_calls(tables, call_sent, call_topics)

print(f"Data version (last pipeline run): {version}")
pd.DataFrame({"rows": {k: len(v) for k, v in tables.items()}})

In [ ]:
# One row per call: enrichment + themes + sentiment, the table every tab starts from
print(all_calls.shape)
all_calls[["date", "call_type", "account", "title", "primary_theme", "sentiment_score",
           "customer_vader", "arc", "teams"]].head()

## 4. Filters and the time period

`sidebar_filters()` draws the sidebar widgets and hands their values to `filter_calls()`, which
does the actual filtering. The quick ranges are derived from the timeline in `src/enrich.py`
(outage start and end, Comply v2 launch), clipped to the data.

Two details matter for new data. When a refresh brings in calls outside the old range, a quick
range moves with the data and a custom range is clipped. Dates are shown, and picked, as mm/dd/yyyy.

In [ ]:
first, last = all_calls["start_time"].min().date(), all_calls["start_time"].max().date()
ranges = app.quick_ranges(first, last)
pd.DataFrame([{"quick range": k, "from": a.strftime(app.DATE_FMT), "to": b.strftime(app.DATE_FMT),
               "calls": len(app.filter_calls(all_calls, a, b))} for k, (a, b) in ranges.items()])

In [ ]:
# The same function the sidebar uses: e.g. support calls about Protect during the outage
a, b = ranges[next(k for k in ranges if k.startswith("Outage"))]
app.filter_calls(all_calls, a, b, types=["support"], product="Protect")[["date", "account", "title", "sentiment_score"]]

## 5. The context: one dict that feeds every tab

`build_context()` takes the filtered calls and computes everything the tabs need, using the same
functions as the pipeline (`account_health`, `incident_timeline`, `competitor_mentions`,
`talk_time`, `monthly_availability`). That is how account risk, SLA and the rest follow the period.

Below it's built for **All time** with the default 99.5% SLA target. Change `START`/`END` (or
`TYPES`, `PRODUCT`, `ACCOUNT`) and re-run the rest of the notebook to see any other view.

In [ ]:
START, END = ranges["All time"]
TYPES, PRODUCT, ACCOUNT = app.CALL_TYPES, "All products", ""
SLA = app.SLA_TARGET

calls = app.filter_calls(all_calls, START, END, TYPES, PRODUCT, ACCOUNT)
ctx = app.build_context(tables, register, ins, calls, START, END, SLA)
pd.DataFrame([{"key": k, "type": type(v).__name__, "size": len(v) if hasattr(v, "__len__") else v}
              for k, v in ctx.items()])

## 6. Overview tab

The Overview opens with a **headline** and two lists, **accomplishments** and **areas to
improve**. None of this text is written by hand or by an LLM: it's generated by rules from the
context, so it changes with the period. For example, choose *Pre-outage* and the outage sentence
disappears.

In [ ]:
display(HTML(f"<p style='font-size:1.05em'>{app.headline_insight(calls, START, END, ctx['sla'])}</p>"))
display(HTML("<b>Accomplishments</b><ul>" + "".join(f"<li>{t}</li>" for t in
             app.accomplishments(calls, ctx["km"], ctx["register"], ctx["health"])) + "</ul>"))
display(HTML("<b>Areas to improve</b><ul>" + "".join(f"<li>{t}</li>" for t in
             app.improvements(calls, ctx["register"], ctx["health"], ctx["sla"], ctx["comp"], ctx["asks"])) + "</ul>"))

**Confidence by call type over time.** Weekly points for periods up to 45 days, monthly
otherwise (`bucket()`), with the outage window shaded.

In [ ]:
app.confidence_by_type_chart(calls, START, END)

**Outage duration and SLA impact by month.** Downtime comes from the incident register; the
allowance is the month's hours × (1 − SLA target). A breached month is drawn in the critical colour.

In [ ]:
display(ctx["sla"][["label", "outage_hours", "allowed_hours", "pct_time_down", "sla_breached", "incidents"]])
app.sla_chart(ctx["sla"])

In [ ]:
app.account_heatmap(calls, START, END)

## 7. Topics tab

Themes come from `data/processed/<mode>/call_topics.parquet` (see `02_topics`). The tab shows
how often each theme comes up by period, how confident calls about each theme are, and the
primary theme by call type.

In [ ]:
app.theme_counts(calls).to_frame('calls mentioning')

In [ ]:
app.theme_period_heatmap(calls)

In [ ]:
app.confidence_by_theme_chart(calls)

In [ ]:
app.primary_theme_by_type_chart(calls)

## 8. Confidence tab

"Confidence" is the vendor's 1–5 call score. The dashboard checks it against its own reading of
the transcript text (`03_sentiment`) without naming the scoring library, so the labels stay in
plain language.

In [ ]:
app.vendor_vs_transcript_chart(calls)

In [ ]:
app.side_by_period_chart(calls)

In [ ]:
# The slice selector: any of call type, month, period, product, theme
fig, table = app.slice_chart(calls, "product")
display(table)
fig

In [ ]:
app.arc_chart(calls)

## 9. Accounts tab

Accounts are ranked by the transparent risk score from `src/insights.py` (see `04_insights`),
recomputed for the period. Clicking an account name in the dashboard opens `account_brief()`: a
Markdown summary plus the account's action items, which can be copied or downloaded.

In [ ]:
h = ctx["health"]
display(h["risk_tier"].value_counts().reindex(app.TIERS).to_frame("accounts"))
app.risk_ranking_chart(h)

In [ ]:
ACCOUNT_NAME = h.iloc[0]["account"]          # the highest-risk account; any name from h works
brief, items = app.account_brief(ACCOUNT_NAME, ctx)
display(Markdown(brief))
items

In [ ]:
app.account_history_chart(calls[calls["account"] == ACCOUNT_NAME], START, END)

## 10. Incidents tab

The register (from `src/incidents.py`) lists each war-room incident and support case with
reported, owned and resolved times. Only incidents reported in the period are shown. The **Call
out** button opens a demo dialog that addresses people by handle; nothing is sent or stored.

In [ ]:
reg = ctx["register"].assign(reported=lambda d: d["reported_at"].map(app.fmt_ts),
                             resolved=lambda d: d["resolved_at"].map(app.fmt_ts))
print(reg["status"].value_counts().to_dict())
reg[["incident_id", "status", "priority", "category", "title", "account", "owner", "reported", "resolved"]].head(10)

In [ ]:
# How the Call Out dialog addresses people
open_inc = reg[reg["status"] == "Open"]
[app.handle(n) for n in open_inc["owner"].dropna().unique()[:5]]

In [ ]:
app.incident_chart(ctx['timeline'], START, END)

## 11. Insights tab

Feature asks, competitor intelligence, talk time and action-item workload. Feature-ask counts
are recounted from the moments in the period; their names come from the saved clusters.

In [ ]:
ctx['asks'].head(8)

In [ ]:
comp = ctx["comp"]
comp.groupby("competitor")["call_id"].nunique().sort_values(ascending=False).to_frame("calls") if len(comp) else "No competitor mentions"

In [ ]:
app.talk_bucket_chart(ctx['talk'])

## 12. Tool Telemetry tab

How well the tooling works and what it costs. LLM spend comes from
`data/processed/llm_usage.jsonl`, which `src/llm.py` appends to on every API call (cache hits
cost nothing and aren't logged). The month's bar turns red once spend reaches the warning share
of the monthly budget, both set in `.env` (`LLM_MONTHLY_BUDGET_USD`, `LLM_BUDGET_WARN_SHARE`).

In [ ]:
usage = app.load_usage()
print(f"Budget {app.money(app.MONTHLY_BUDGET_USD)}/month, red from {app.BUDGET_WARN_SHARE:.0%}; "
      f"{len(usage)} logged API calls, {app.money(usage['cost_usd'].sum()) if len(usage) else '$0'} in total")
display(usage.groupby("step")["cost_usd"].agg(["count", "sum"]).round(4) if len(usage) else None)
app.monthly_cost_chart(usage, app.MONTHLY_BUDGET_USD, app.BUDGET_WARN_SHARE) if len(usage) else None

In [ ]:
pd.Series({k: v for k, v in topic_metrics.items() if not isinstance(v, (dict, list))}, name="topic quality")

## 13. Every number follows the period

`build_context()` is run for each quick range and the main KPIs are put side by side. In the
dashboard this is what happens when you change the time period.

In [ ]:
rows = []
for name, (a, b) in ranges.items():
    c = app.filter_calls(all_calls, a, b)
    if c.empty:
        continue
    x = app.build_context(tables, register, ins, c, a, b, SLA)
    rows.append({
        "period": name, "calls": len(c),
        "avg confidence": round(c["sentiment_score"].mean(), 2),
        "high-risk accounts": int((x["health"]["risk_tier"] == "High").sum()),
        "incidents reported": len(x["register"]),
        "still open": int((x["register"]["status"] == "Open").sum()),
        "outage hours": x["sla"]["outage_hours"].sum(),
        "SLA breached": bool(x["sla"]["sla_breached"].any()),
        "competitor calls": x["comp"]["call_id"].nunique() if len(x["comp"]) else 0,
    })
pd.DataFrame(rows).set_index("period")

**What this shows.** Risk depends on the window: when this was written, ten accounts were *High*
risk over all time but only one in the pre-outage period, because the score only sees that
window's calls. The SLA view only counts months inside
the period. That's deliberate (the dashboard answers "what happened in this period?"), but worth
knowing when comparing screenshots taken with different filters.

## 14. New transcripts, refresh and logs

The sidebar's Data section runs `transcript_watch()` as a Streamlit **fragment** on a timer
(every minute by default; ⚙ Config sets 30 s, 5 min or off). It only **scans and notifies**: it
never reloads the page or runs the pipeline by itself, so nobody's view changes under them.

| Step | Function | What happens |
|---|---|---|
| Scan | `src.pipeline.scan_raw()` | Lists folders in `data/raw/` not yet processed. A folder must be unchanged for 30 s (still copying otherwise), and unreadable ones are reported |
| Notify | `transcript_watch()` | One pop-up per new batch, plus a sidebar list of the titles; keyword hits are marked *possibly not business* |
| Ignore | `new_transcript_list()` → `src.screen.exclude_call()` | Adds a new transcript to `data/exclusions.csv` and moves its folder from `data/raw/` to `data/exclude/`; it's never processed |
| Refresh | `run_refresh()` | On **Refresh dashboard data**: runs the full pipeline (including the non-business screen) under a lock (one run at a time across tabs), shows the log live, clears the cache, reloads |
| Newer data | data version check | If another tab or the command line produced newer data, offers **Load latest data** |
| Logs | `src.pipeline.capture_logs()` | Every run is appended to `outputs/logs/pipeline.log`, also shown under **Pipeline log** |

In [ ]:
from src.pipeline import scan_raw, last_run, read_log_tail, PIPELINE_LOG
scan = scan_raw()
run = last_run()
print("New:", scan["new"] or "none", "| arriving:", scan["arriving"] or "none", "| unreadable:", scan["unreadable"] or "none")
print("Possibly not business:", scan["flagged"] or "none", "| excluded so far:", len(app.excluded_ids()))
print("Last run finished:", pd.Timestamp(run["finished_at"]).strftime(app.DATETIME_FMT), "UTC · modes:", run.get("modes_built"))
print(f"\nLast lines of {PIPELINE_LOG}:\n" + read_log_tail(8))

In [ ]:
# The exclusion list: what was kept out, who decided and why (Ignore in the dashboard = source "user")
from src.screen import load_exclusions, EXCLUSIONS
print(EXCLUSIONS)
load_exclusions()

## 15. Extending the dashboard

**Add a chart.** Write a function that takes DataFrames and returns a Plotly figure (pass it
through `_style()` for the shared layout), try it in a cell here, then call
`st.plotly_chart(fig, width="stretch")` from the right `render_*` function.

**Add a tab.** Write `render_mytab(ctx)`, add its name to the `st.tabs([...])` list in `main()`,
and call it in a new `with tabs[n]:` block. If it needs a new per-period number, add it to
`build_context()` so this notebook and the dashboard share it.

**Conventions to keep.**

- Fixed colours per call type (`CALL_TYPE_COLORS`). Status colours (`GOOD`, `WARNING`,
  `CRITICAL`) are only for state: risk tiers, SLA breaches, budget.
- Dates as mm/dd/yyyy (`DATE_FMT`, `DATETIME_FMT`, `PLOTLY_DATE`). Tables go through
  `show_table()`, which also formats date columns and labels internal meetings.
- Plain-language labels: no library or model names in the UI.
- Nothing slow or paid in the app. Anything that needs an LLM or a model belongs in the pipeline,
  and the app reads its output.

**After changing `app.py`:** a running `streamlit run` session picks the change up on save
(*Rerun* in the top-right corner). The .exe has its own copy of the code and needs a rebuild
(`.\build_exe.ps1`).